# Module 3: Model Development — Ejercicios Prácticos

**Objetivos:**
- Entrenar Logistic Regression, Decision Tree, Random Forest, XGBoost
- Comparar modelos
- Tuning de hiperparámetros

**Tiempo:** 30 min

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

## Ejercicio 1: Cargar Dataset y Preparar

In [ ]:
# Dataset: Breast Cancer (clasificación binaria)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Clases: {np.unique(y)}")
print(f"Distribución: {np.bincount(y)}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Ejercicio 2: Entrenar Múltiples Modelos

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=0)
}

results = []

for name, model in models.items():
    print(f"\nEntrenando {name}...")
    
    # Entrenar
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'AUC-ROC': auc
    })
    
    print(f"  Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

# Tabla comparativa
df_results = pd.DataFrame(results)
print("\n📊 Comparación de Modelos:")
print(df_results.to_string(index=False))

## Ejercicio 3: Visualizar Resultados

In [ ]:
# Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Métricas por modelo
df_results.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Comparación de Modelos (Métricas)')
axes[0].set_ylabel('Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0.9, 1.0])

# AUC-ROC
df_results.set_index('Model')['AUC-ROC'].plot(kind='bar', ax=axes[1], color='skyblue', edgecolor='black')
axes[1].set_title('ROC-AUC Score')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_ylim([0.9, 1.0])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Ejercicio 4: Tuning de Random Forest (GridSearch)

In [ ]:
print("Tuning Random Forest con GridSearchCV...\n")

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(rf, params, cv=5, scoring='f1', verbose=0, n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV F1 score: {grid.best_score_:.4f}")

# Evaluar en test
best_rf = grid.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_pred_proba_best = best_rf.predict_proba(X_test)[:, 1]

print(f"\nTest Metrics (Best Model):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  F1: {f1_score(y_test, y_pred_best):.4f}")
print(f"  AUC-ROC: {roc_auc_score(y_test, y_pred_proba_best):.4f}")

## Ejercicio 5: Feature Importance

In [ ]:
# Feature importance de Random Forest
importances = best_rf.feature_importances_
feature_names = X.columns

# Top 10 features
indices = np.argsort(importances)[::-1][:10]
top_features = [(feature_names[i], importances[i]) for i in indices]

print("Top 10 Most Important Features:")
for feature, importance in top_features:
    print(f"  {feature}: {importance:.4f}")

# Plotear
plt.figure(figsize=(10, 6))
feature_list = [f[0] for f in top_features]
importance_list = [f[1] for f in top_features]
plt.barh(range(len(feature_list)), importance_list, edgecolor='black')
plt.yticks(range(len(feature_list)), feature_list)
plt.xlabel('Feature Importance')
plt.title('Top 10 Features (Random Forest)')
plt.tight_layout()
plt.show()

## Resumen

In [ ]:
print("="*60)
print("RESUMEN: MODEL DEVELOPMENT")
print("="*60)
print()
print("BASELINE → SIMPLE → COMPLEX")
print()
print("1. Logistic Regression:")
print("   - Rápido, interpretable, baseline")
print()
print("2. Decision Tree:")
print("   - Interpretable, no requiere scaling")
print()
print("3. Random Forest:")
print("   - Balance accuracy + velocidad")
print()
print("4. XGBoost:")
print("   - Mejor accuracy, requiere tuning")
print()
print("ESTRATEGIA: Empezar simple, validar, después complejo")
print("="*60)